<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_12_Idempotent_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
sample_data = [
    (1, "Jon King", "engineer", 80000.0),
    (2, "Sarah Kim", "Manager", 90000.0)
]

In [ ]:
import sqlite3
import logging

logger = logging.getLogger(__name__)

def create_database_and_table(db_path: str = "employees_data.db") -> None:
    """Create the database and employees table if they do not exist."""

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS employees (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                role TEXT,
                salary REAL
            )
        """)

# Calling function
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    create_database_and_table()
    logger.info("Database and employees table are ready.")

## **Non-Idempotent version**

In [ ]:
# def insert_employees_non_idempotent(data: list, db_path: str = "employees_data.db") -> None:
#     """This version is NOT idempotent — rerunning causes issues."""
#     with sqlite3.connect(db_path) as conn:
#         cursor = conn.cursor()

#         for emp in data:
#             cursor.execute("""
#                 INSERT INTO employees (id, name, role, salary)
#                 VALUES (?, ?, ?, ?)
#             """, emp)

#         conn.commit()
#         logger.info(f"Inserted {len(data)} employee records (non-idempotent).")

# if __name__ == "__main__":
#     insert_employees_non_idempotent(sample_data)

## **Idempotent version**

In [ ]:
def insert_employees_idempotent(data: list, db_path: str = "employees_data.db") -> None:
    """Idempotently insert employee records using an UPSERT."""

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        for emp in data:
            cursor.execute("""
                INSERT INTO employees (id, name, role, salary)
                VALUES (?, ?, ?, ?)
                ON CONFLICT(id) DO UPDATE SET
                    name = excluded.name,
                    role = excluded.role,
                    salary = excluded.salary
            """, emp)
        conn.commit()

    logger.info(f"Upserted {len(data)} employee records (idempotent).")


# Calling the function
if __name__ == "__main__":
    insert_employees_idempotent(sample_data)

INFO: Upserted 2 employee records (idempotent).


## **Make transformations pure and deterministic**

In [ ]:
# NOT idempotent: accumulates on each run
running_total = 0

def add_to_total(value: int) -> int:
    global running_total
    running_total += value
    return running_total

In [ ]:
from typing import List
# Idempotent: pure function
def calculate_total(values: List[int]) -> int:
    return sum(values)

## **Pure and idempotent**

In [ ]:
import pandas as pd
from typing import Dict

raw_df = pd.DataFrame([
    {'transaction_id': 1, 'user_id': 'u123', 'amount': 100, 'currency': 'USD'},
    {'transaction_id': 2, 'user_id': 'u456', 'amount': 50, 'currency': 'EUR'},
])

# Dictionaries of rates and tiers
rates = {'USD': 1.0, 'EUR': 1.09}
cashback_rates = {'gold': 0.05, 'silver':0.025 }
tiers = {'u123': 'gold', 'u456': 'basic'}

def enrich_transformations(
        df: pd.DataFrame,
        rates:Dict[str,float],
        tiers:Dict[str,str],
        cashback_rates:Dict[str, float]
        ) -> pd.DataFrame:

    """Return an enriched copy of the transaction DataFrame"""
    # Create a copy of the DataFrame
    df = df.copy()

    df["exchange_rate"] = df["currency"].map(rates).fillna(1.0)
    df["user_tier"]= df["user_id"].map(tiers)

    # Convert each transaction amount to USD.
    df["amount_usd"] = df["amount"] * df["exchange_rate"]

     # Calculate cashbacks rates
    df["cashback_rate"] = df["user_tier"].map(cashback_rates).fillna(0.0)
    df["cashback"] = df["amount_usd"] * df["cashback_rate"]

    return df

# Call the function
if __name__ == "__main__":
    results_df = enrich_transformations(raw_df, rates, tiers,
                                        cashback_rates)
    print(results_df)

   transaction_id user_id  amount currency  exchange_rate user_tier  \
0               1    u123     100      USD           1.00      gold   
1               2    u456      50      EUR           1.09     basic   

   amount_usd  cashback_rate  cashback  
0       100.0           0.05       5.0  
1        54.5           0.00       0.0  


## **Adding Explicit checks before side effects**

In [ ]:
import pandas as pd
import logging
from datetime import datetime

# data
clients = pd.DataFrame([
    {'user_id': 1, 'name': 'Leanne Graham', 'notification': None},
    {'user_id': 5, 'name': 'Chelsey Dietrich',   'notification': None},
])

logger = logging.getLogger(__name__)

def send_notification_if_needed(
    df: pd.DataFrame,
    user_id: int,
    message: str = "Your report is ready"
) -> pd.DataFrame:
    """
    Send notification only if it hasn't been sent already.
    """

    # Check if notification was already sent for this user
    already_notified = df.loc[
        df['user_id'] == user_id, 'notification'].notna().any()
    if already_notified:
        logger.info(f"Notification for user {user_id} already sent. Skipping.")
        return df  # Return unchanged df

    logger.info(f"Sending notification to user {user_id}: {message}")
    # Update the notification status
    df.loc[df['user_id'] == user_id, 'notification'] = (
        f"Message sent at {datetime.now().isoformat()}"
    )
    return df

In [ ]:
# First call — sends the notification
clients = send_notification_if_needed(clients, 5)
print(clients)

INFO: Sending notification to user 5: Your report is ready


   user_id              name                                notification
0        1     Leanne Graham                                        None
1        5  Chelsey Dietrich  Message sent at 2026-08-26T10:39:15.456249


In [ ]:
# Second call — detects existing notification and skips
clients = send_notification_if_needed(clients, 5)
print(clients)

INFO: Notification for user 5 already sent. Skipping.


   user_id              name                                notification
0        1     Leanne Graham                                        None
1        5  Chelsey Dietrich  Message sent at 2026-08-26T10:39:15.456249


## **Combining Idempotency with Retries**

In [ ]:
# data
clients = pd.DataFrame([
    {'user_id': 1, 'name': 'Leanne Graham', 'notification': None},
    {'user_id': 5, 'name': 'Chelsey Dietrich',   'notification': None},
])

In [ ]:
import uuid
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

logger = logging.getLogger(__name__)

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type(requests.exceptions.Timeout)
)
def send_notification(user_id: int, message: str, request_id: str) -> None:
    """Send notification safely using a unique transaction request_id."""

    # Key is unique per message event, but identical across retries
    idempotency_key = f"notification:{user_id}:{request_id}"

    response = requests.post(
         "https://httpbin.org/post",
        json={"user_id": user_id, "message": message},
        headers={"Idempotency-Key": idempotency_key},
        timeout=5
    )
    response.raise_for_status()

def send_notification_if_needed(df: pd.DataFrame,
                                user_id: int, message: str = "Your report is ready!"
                                ) -> pd.DataFrame:
    """Send notification only if it hasn't been sent already"""

    already_notified = df.loc[df["user_id"] == user_id, "notification"].notna().any()

    if already_notified:
        logger.info(f"Notification for user ID {user_id} already sent. Skipping.")
        return df

    logger.info(f"Attempting to notify user ID: {user_id}...")

    # Generate the unique token BEFORE entering the network retry loop
    tx_id = str(uuid.uuid4())
    send_notification(user_id, message, request_id=tx_id)

    df.loc[df["user_id"] == user_id, "notification"] = datetime.now().isoformat()
    return df

In [ ]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    results = send_notification_if_needed(clients, user_id=5)
    logger.info(results)

INFO: Attempting to notify user ID: 5...
INFO:    user_id              name                notification
0        1     Leanne Graham                        None
1        5  Chelsey Dietrich  2026-08-26T10:39:17.792505


## **Challenge**

In [ ]:
sent_notifications = set()

def send_notification(user_id: int, message: str) -> None:
    """Send a notification only once for each unique message."""

    notification_key = (user_id, message)

    if notification_key not in sent_notifications:
        print(f"Sending notification to user {user_id}: {message}")
        sent_notifications.add(notification_key)
    else:
        print(f"Notification already sent to user {user_id}: {message}")

## **Solution**

In [ ]:
import sqlite3
import logging

logger = logging.getLogger(__name__)

def create_database(db_name: str = "notifications.db") -> sqlite3.Connection:
    """Create the database and notifications table if they do not exist."""
    conn = sqlite3.connect(db_name)
    with conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS sent_notifications (
                user_id INTEGER,
                message TEXT,
                PRIMARY KEY (user_id, message)
            )
        """)
    return conn


def execute_network_delivery(user_id: int, message: str) -> None:
     """Deliver the message (e.g., Email, SMS, or API POST)."""
     logger.info(f"Sending notification to user {user_id}: {message}")


def send_notification_if_needed(conn: sqlite3.Connection, user_id: int, message: str) -> None:
    """Check the database, send the message, and save the result."""

    # Check if notification has already been sent
    cursor = conn.execute("""
        SELECT 1
        FROM sent_notifications
        WHERE user_id = ? AND message = ?
    """, (user_id, message))

    if cursor.fetchone():
        logger.info(f"Notification already sent to user {user_id}: {message}")
        return

    # call the function to send notfication
    execute_network_delivery(user_id, message)

    # Record the successful event in a managed database transaction
    try:
        with conn:
            conn.execute("""
                INSERT INTO sent_notifications (user_id, message)
                VALUES (?, ?)
            """, (user_id, message))

    except sqlite3.IntegrityError:
        logger.warning(f"Race condition caught. Record already verified for user {user_id}.")


# Execute function
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    # Creating database
    connection = create_database()

    user = 101
    alert_msg = "Your security code is 4821."

    # Execution 1
    send_notification_if_needed(connection, user_id=user, message=alert_msg)

    # Execution 2
    send_notification_if_needed(connection, user_id=user, message=alert_msg)

    connection.close()

INFO: Sending notification to user 101: Your security code is 4821.
INFO: Notification already sent to user 101: Your security code is 4821.
